# Mini Project 1 — Analysis Notebook

**Your name: Nicolas Rodriguez**  
**Dataset: Spotify API**  
**Date: May 11, 2026**  

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** *(What is it? Where did it come from? Paste the URL or citation from your MP1a submission.)*

**Why this dataset:** *(One sentence connecting it to your HCD work or research interests.)*

**Three analytical questions:**

1. *How much overlap exists between my top artists across a short term (4 weeks) medium term (6 months) and long term (since having the account), what does this reveal about wether my taste is stable or shifting?* 
2. *What is the release year distribution of my top tracks across time ranges, do I gravitate to an older catalog or newer releases?* 
3. *From my 50 most recently played tracks, what proportion of listens come from albums vs playlist vs other contexts, which artists do I listen to most in album mode vs playlist mode?*

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [2]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual filename.
# The file should be in the same folder as this notebook.
# If you're loading from an API result, replace pd.read_csv() with the appropriate call.
#
# Example (app review dataset from class):
# df = pd.read_csv('app_reviews_demo.csv')

import pandas as pd

df = pd.read_csv('Spotify NR data.csv')  # replace with your filename

print(df.shape)
df.head()

(110, 13)


,source,time_range_key,rank,artist,name,album,release_year,duration_sec,explicit,time_range,track,played_at,context
0,top_artists,short_term,1.0,Pink Floyd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,top_artists,short_term,2.0,Yandel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,top_artists,short_term,3.0,Feid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,top_artists,short_term,4.0,Radiohead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,top_artists,short_term,5.0,Omar Courtz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   source          110 non-null    str    
 1   time_range_key  30 non-null     str    
 2   rank            30 non-null     float64
 3   artist          110 non-null    str    
 4   name            30 non-null     str    
 5   album           30 non-null     str    
 6   release_year    80 non-null     float64
 7   duration_sec    30 non-null     float64
 8   explicit        30 non-null     object 
 9   time_range      30 non-null     str    
 10  track           50 non-null     str    
 11  played_at       50 non-null     str    
 12  context         50 non-null     str    
dtypes: float64(3), object(1), str(9)
memory usage: 11.3+ KB


In [4]:
# Summary statistics for numeric columns
df.describe()

,rank,release_year,duration_sec
count,30.000000,80.000000,30.000000
mean,5.500000,2017.375000,277.200000
std,2.921384,13.300209,194.099583
min,1.000000,1975.000000,104.000000
25%,3.000000,2017.000000,169.250000
50%,5.500000,2021.000000,213.000000
75%,8.000000,2025.000000,280.500000
max,10.000000,2026.000000,813.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** How do my top artist rankings change across short-, medium-, and long-term listening windows?

In [5]:
# Script 5 + Script 6 adapted for this notebook.
# Build top-10 artist lists by time range, then compare overlap across windows.

from IPython.display import display

required_cols = {"source", "artist", "time_range_key", "rank"}
if "df" not in globals() or not required_cols.issubset(df.columns):
    raise NameError("Run the dataset loading cell first so df is available.")

artists_source = (
    df.loc[df["source"] == "top_artists", ["artist", "time_range_key", "rank"]]
    .dropna(subset=["artist", "time_range_key", "rank"])
    .copy()
)
artists_source["rank"] = artists_source["rank"].astype(int)

# Keep this dataframe for visualization in Section 4.
time_order = ["short_term", "medium_term", "long_term"]
artists_rank_df = artists_source.copy()
artists_rank_df["time_range_key"] = pd.Categorical(
    artists_rank_df["time_range_key"], categories=time_order, ordered=True
)
artists_rank_df = artists_rank_df.sort_values(["artist", "time_range_key"])

# Script 5 behavior: collect top artists for each time window.
time_ranges = {
    "short_term": "Last 4 Weeks",
    "medium_term": "Last 6 Months",
    "long_term": "All Time",
}

top_artists = {}
for range_key in time_ranges:
    artists = (
        artists_source.loc[artists_source["time_range_key"] == range_key]
        .sort_values("rank")["artist"]
        .head(10)
        .tolist()
    )
    top_artists[range_key] = artists

# Table 1: Top artists by rank across time windows.
rank_table = (
    artists_rank_df
    .pivot_table(index="rank", columns="time_range_key", values="artist", aggfunc="first")
    .rename(columns=time_ranges)
    .sort_index()
)
rank_table.index.name = "Rank"

# Script 6 behavior: overlap and listener profile.
short = set(top_artists["short_term"])
medium = set(top_artists["medium_term"])
long = set(top_artists["long_term"])

short_medium = short & medium
medium_long = medium & long
all_three = short & medium & long
only_short = short - medium - long

if len(all_three) >= 6:
    profile = "Loyalist - you return to the same artists consistently"
elif len(all_three) >= 3:
    profile = "Balanced - mix of core favorites and new discoveries"
else:
    profile = "Explorer - your taste shifts significantly over time"

# Table 2: Overlap summary and listener profile.
overlap_summary = pd.DataFrame(
    {
        "Metric": [
            "Overlap: short vs medium",
            "Overlap: medium vs long",
            "Overlap: all three",
            "Listener profile",
        ],
        "Value": [
            f"{len(short_medium)}/10",
            f"{len(medium_long)}/10",
            f"{len(all_three)}/10",
            profile,
        ],
    }
)

# Table 3: Which artists are core vs newly appearing.
artist_grouping = pd.DataFrame(
    {
        "Core artists (all three windows)": pd.Series(sorted(all_three)),
        "New discoveries (short-term only)": pd.Series(sorted(only_short)),
    }
)

print("Top artists by rank across listening windows")
display(rank_table)

print("\nOverlap summary")
display(overlap_summary)

print("\nCore artists vs new discoveries")
display(artist_grouping)


Top artists by rank across listening windows


time_range_key,Last 4 Weeks,Last 6 Months,All Time
Rank,,,
1,Pink Floyd,Pink Floyd,Pink Floyd
2,Yandel,Feid,Feid
3,Feid,Radiohead,Radiohead
4,Radiohead,Omar Courtz,Mac DeMarco
5,Omar Courtz,Milo j,Mac Miller
6,Olivia Dean,Mac DeMarco,The Doors
7,Trueno,Olivia Dean,Bad Bunny
8,Harry Styles,Brent Faiyaz,Brent Faiyaz
9,Lenny Kravitz,Yandel,Jeff Buckley



Overlap summary


,Metric,Value
0,Overlap: short vs medium,6/10
1,Overlap: medium vs long,5/10
2,Overlap: all three,3/10
3,Listener profile,Balanced - mix of core favorites and new disco...



Core artists vs new discoveries


,Core artists (all three windows),New discoveries (short-term only)
0,Feid,Harry Styles
1,Pink Floyd,Lenny Kravitz
2,Radiohead,Nsqk
3,NaN,Trueno


**Interpretation:**  
The rank table shows both stability and change across listening windows. Pink Floyd remains at rank #1 in short-, medium-, and long-term lists, suggesting a consistent listening anchor. Feid and Radiohead also stay near the top, while artists like Yandel are strong in the short term but less prominent in long-term rankings. This suggests my listening includes a stable core of artists plus more recent rotations that shift over shorter windows.

**Question 2:** *What is the release year distribution of my top tracks across time ranges, do I gravitate to an older catalog or newer releases?*

In [6]:
# Script 8 adapted for this notebook.
# Analyze release-year distribution of top tracks across time ranges.

from IPython.display import display
import plotly.express as px

required_cols = {"source", "name", "artist", "album", "release_year", "time_range"}
if "df" not in globals() or not required_cols.issubset(df.columns):
    raise NameError("Run the dataset loading cell first so df is available.")

# Pull only top-track records from the exported Spotify dataset.
df_tracks = (
    df.loc[df["source"] == "top_tracks", ["name", "artist", "album", "release_year", "duration_sec", "explicit", "time_range"]]
    .dropna(subset=["name", "artist", "release_year", "time_range"])
    .copy()
)

# Keep windows in chronological scope order.
window_order = ["Last 4 Weeks", "Last 6 Months", "All Time"]
df_tracks["time_range"] = pd.Categorical(df_tracks["time_range"], categories=window_order, ordered=True)

df_tracks["release_year"] = df_tracks["release_year"].astype(int)
df_tracks["decade"] = (df_tracks["release_year"] // 10) * 10

# Summary table by listening window.
year_summary = (
    df_tracks.groupby("time_range", as_index=False)
    .agg(
        tracks_count=("name", "count"),
        avg_release_year=("release_year", "mean"),
        oldest_release_year=("release_year", "min"),
        newest_release_year=("release_year", "max"),
    )
    .sort_values("time_range")
)
year_summary["avg_release_year"] = year_summary["avg_release_year"].round(0).astype(int)

# Counts by exact year and by decade (used for Q2 visualizations).
year_distribution = (
    df_tracks.pivot_table(
        index="release_year",
        columns="time_range",
        values="name",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(columns=window_order)
    .sort_index()
)

decade_distribution = (
    df_tracks.pivot_table(
        index="decade",
        columns="time_range",
        values="name",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(columns=window_order)
    .sort_index()
)

decade_long = (
    df_tracks.groupby(["time_range", "decade"], as_index=False)
    .size()
    .rename(columns={"size": "tracks_count"})
    .sort_values(["decade", "time_range"])
)

print("Release-year summary by listening window")
display(year_summary)

print("\nTrack count by release year and window")
display(year_distribution)

print("\nTrack count by decade and window")
display(decade_distribution)


Release-year summary by listening window


,time_range,tracks_count,avg_release_year,oldest_release_year,newest_release_year
0,Last 4 Weeks,10,2020,1975,2026
1,Last 6 Months,10,2019,1975,2026
2,All Time,10,2005,1975,2025



Track count by release year and window


time_range,Last 4 Weeks,Last 6 Months,All Time
release_year,,,
1975,1,1,1
1979,0,0,2
1998,0,0,1
2002,0,0,1
2017,1,0,0
2019,0,1,0
2020,0,1,0
2024,0,0,4
2025,0,4,1



Track count by decade and window


time_range,Last 4 Weeks,Last 6 Months,All Time
decade,,,
1970,1,1,3
1990,0,0,1
2000,0,0,1
2010,1,1,0
2020,8,8,5


**Interpretation:**  
(See the combined visualization rationale in Section 4 for the final chart-based interpretation.)

**Question 3:** *From my 50 most recently played tracks, what proportion of listens come from albums vs playlist vs other contexts, which artists do I listen to most in album mode vs playlist mode?*

In [11]:
# Script 10 adapted for this notebook.
# Analyze 50 recent plays: context breakdown and top artists by context.

from IPython.display import display

required_cols = {"source", "track", "artist", "played_at", "context", "release_year"}
if "df" not in globals() or not required_cols.issubset(df.columns):
    raise NameError("Run the dataset loading cell first so df is available.")

# Pull only recently played records from the exported Spotify dataset.
df_recent = (
    df.loc[df["source"] == "recently_played", ["track", "artist", "played_at", "context", "release_year"]]
    .dropna(subset=["track", "artist", "played_at"])
    .copy()
)

df_recent["played_at"] = pd.to_datetime(df_recent["played_at"], errors="coerce")
df_recent["context"] = df_recent["context"].fillna("unknown")
df_recent = df_recent.sort_values("played_at", ascending=False)

# Table 1: listening context breakdown.
context_summary = (
    df_recent["context"]
    .value_counts(dropna=False)
    .rename_axis("context")
    .reset_index(name="plays")
)
context_summary["percent"] = (context_summary["plays"] / len(df_recent) * 100).round(1)

# Table 2: top artists overall in recent plays.
top_artists_recent = (
    df_recent["artist"]
    .value_counts()
    .head(10)
    .rename_axis("artist")
    .reset_index(name="plays")
)

# Table 3: top artists within album vs playlist context.
focus_contexts = ["album", "playlist"]
artist_by_context = (
    df_recent[df_recent["context"].isin(focus_contexts)]
    .groupby(["context", "artist"], as_index=False)
    .size()
    .rename(columns={"size": "plays"})
    .sort_values(["context", "plays"], ascending=[True, False])
)
artist_by_context = artist_by_context.groupby("context").head(5)

# Quick recency check (most recent and oldest track in this sample).
most_recent = df_recent.iloc[0]
oldest = df_recent.iloc[-1]

print("Listening context breakdown (recent plays)")
display(context_summary)

print("\nTop artists in recent plays")
display(top_artists_recent)

print("\nTop artists by listening context (album vs playlist)")
display(artist_by_context)

print(
    f"\nMost recent listen: {most_recent['track']} by {most_recent['artist']}\n"
    f"Oldest in sample: {oldest['track']} by {oldest['artist']}"
)


Listening context breakdown (recent plays)


,context,plays,percent
0,playlist,24,48.0
1,album,14,28.0
2,unknown,12,24.0



Top artists in recent plays


,artist,plays
0,Daft Punk,6
1,The Strokes,6
2,Joy Rhodes,2
3,Feid,2
4,Central Cee,2
5,Pale Jay,2
6,Bombay Bicycle Club,1
7,Daryl Hall & John Oates,1
8,Gilsons,1
9,HEAVY CHEST,1



Top artists by listening context (album vs playlist)


,context,artist,plays
4,album,The Strokes,6
1,album,Daft Punk,5
0,album,Bombay Bicycle Club,1
2,album,Daryl Hall & John Oates,1
3,album,Gilsons,1
5,playlist,Bird and Byron,1
6,playlist,Cloak Bay,1
7,playlist,Daft Punk,1
8,playlist,Dal,1
9,playlist,Faye Meana,1



Most recent listen: The Game of Love by Daft Punk
Oldest in sample: Lose Yourself to Dance (feat. Pharrell Williams) by Daft Punk


**Interpretation:**  
The donut chart shows that playlists account for the largest share of my recent listening, with album listens as the second-largest mode and a smaller unknown-context segment. This suggests I rely on playlists for most day-to-day listening while still spending meaningful time in album-focused sessions. The artist-by-context bar chart adds detail: the leading artists in album mode and playlist mode are not identical, which means my behavior changes based on how I listen. Together, these visuals show both the overall listening split and how artist preference shifts by context.

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [8]:
# Artist rank movement across listening windows (connected dot plot)
import plotly.express as px

if "artists_rank_df" not in globals():
    raise NameError("Run the Question 1 analysis cell first to create artists_rank_df.")

focus_artists = (
    artists_rank_df.groupby("artist")["rank"]
    .min()
    .sort_values()
    .head(10)
    .index
)

plot_df = (
    artists_rank_df[artists_rank_df["artist"].isin(focus_artists)]
    .dropna(subset=["time_range_key", "rank"])
    .sort_values(["artist", "time_range_key"])
)

# High-contrast, colorblind-friendly palette for clearer differentiation.
artist_palette = [
    "#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7",
    "#56B4E9", "#F0E442", "#000000", "#8A2BE2", "#A6761D",
]

fig = px.line(
    plot_df,
    x="time_range_key",
    y="rank",
    color="artist",
    markers=True,
    color_discrete_sequence=artist_palette,
    category_orders={"time_range_key": ["short_term", "medium_term", "long_term"]},
    labels={
        "time_range_key": "Listening Window",
        "rank": "Artist Rank (1 = highest)",
        "artist": "Artist",
    },
    title="Top 10 artists shift differently across short, medium, and long-term windows",
)

fig.update_yaxes(autorange="reversed", dtick=1)
fig.update_traces(line=dict(width=3), marker=dict(size=9))

# Put each artist in its own legend group so extra group gap adds spacing.
fig.for_each_trace(lambda trace: trace.update(legendgroup=trace.name))

fig.update_layout(
    template="plotly_white",
    legend=dict(
        title="Artists",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
        font=dict(size=10),
        tracegroupgap=8,
    ),
    margin=dict(r=200),
)
fig.show()


**Chart rationale:**  
A connected dot plot is a better fit for this question because the main goal is to show how each artist's rank moves across ordered time windows. The connecting lines make changes in direction easy to see while still keeping exact rank values visible at each point. Reversing the y-axis keeps rank #1 at the top, matching how rankings are read. The key takeaway is that some artists remain consistently near the top while others change position as the listening window broadens.

For Question 2, the strip plot and grouped decade bar chart work together: the strip plot preserves exact release years in each listening window, while the decade bars summarize broader era shifts. The visuals show that each window is centered on newer releases (late 2010s to 2020s), but older tracks still appear in the mix. This indicates a primarily contemporary listening pattern with a persistent classic-catalog influence over the longer term.

In [9]:
# Q2 Visualization 1: release year strip plot by listening window
if "df_tracks" not in globals() or "window_order" not in globals():
    raise NameError("Run the Question 2 analysis cell first to create df_tracks.")

strip_fig = px.strip(
    df_tracks.sort_values("time_range"),
    x="time_range",
    y="release_year",
    color="time_range",
    category_orders={"time_range": window_order},
    labels={
        "time_range": "Listening Window",
        "release_year": "Release Year",
    },
    title="Top-track release years cluster around newer music, with older tracks still present",
)

# Slightly larger markers to improve readability for small samples.
strip_fig.update_traces(marker=dict(size=11, opacity=0.8), jitter=0.35)

# Add mean release-year marker for each window.
means = df_tracks.groupby("time_range", observed=False)["release_year"].mean().reindex(window_order)
strip_fig.add_scatter(
    x=window_order,
    y=means.values,
    mode="markers+lines",
    marker=dict(symbol="diamond", size=12, color="black"),
    line=dict(color="black", dash="dot"),
    name="Average release year",
)

strip_fig.update_layout(template="plotly_white", showlegend=True)
strip_fig.show()

In [10]:
# Q2 Visualization 2: grouped bar chart by decade and listening window
if "decade_long" not in globals() or "window_order" not in globals():
    raise NameError("Run the Question 2 analysis cell first to create decade_long.")

bars_fig = px.bar(
    decade_long,
    x="decade",
    y="tracks_count",
    color="time_range",
    barmode="group",
    category_orders={"time_range": window_order},
    labels={
        "decade": "Release Decade",
        "tracks_count": "Number of Top Tracks",
        "time_range": "Listening Window",
    },
    title="Recent windows lean heavily toward 2020s tracks, while all-time includes older decades",
)

bars_fig.update_layout(template="plotly_white")
bars_fig.show()

In [12]:
# Save Plotly visualizations as PNG using Kaleido
from pathlib import Path
import importlib
import subprocess
import sys

# Ensure kaleido is available in this kernel.
try:
    importlib.import_module("kaleido")
except ImportError:
    print("Installing kaleido...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kaleido", "-q"])
    except subprocess.CalledProcessError:
        # Fallback for externally managed Python environments.
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "kaleido",
            "-q",
            "--break-system-packages",
        ])

output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)

figures_to_save = [
    ("fig", "q1_artist_rank_movement.png"),
    ("strip_fig", "q2_release_year_strip.png"),
    ("bars_fig", "q2_decade_grouped_bar.png"),
]

saved_files = []
for fig_name, filename in figures_to_save:
    if fig_name in globals():
        out_path = output_dir / filename
        globals()[fig_name].write_image(str(out_path), format="png", scale=2)
        saved_files.append(out_path)
    else:
        print(f"Skipping {fig_name}: figure not found. Run its chart cell first.")

if saved_files:
    print("\nSaved PNG files:")
    for p in saved_files:
        print(f"- {p}")
else:
    print("No figures were saved.")


Saved PNG files:
- figures/q1_artist_rank_movement.png
- figures/q2_release_year_strip.png
- figures/q2_decade_grouped_bar.png


# Question 3


In [12]:
# Q3 Visualization 1: donut chart of recent listening context share
import plotly.express as px

if "context_summary" not in globals():
    raise NameError("Run the Question 3 analysis cell first to create context_summary.")

q3_donut = px.pie(
    context_summary,
    names="context",
    values="plays",
    hole=0.55,
    color="context",
    labels={"context": "Listening Context", "plays": "Plays"},
    title="Recent listening is playlist-led, with album listening as a strong secondary mode",
)

q3_donut.update_traces(
    textposition="inside",
    texttemplate="%{label}<br>%{percent}",
    hovertemplate="Context: %{label}<br>Plays: %{value}<br>Share: %{percent}<extra></extra>",
)
q3_donut.update_layout(template="plotly_white")
q3_donut.show()

In [13]:
# Q3 Visualization 2: top artists by listening mode (album vs playlist)
import plotly.express as px

if "artist_by_context" not in globals():
    raise NameError("Run the Question 3 analysis cell first to create artist_by_context.")

context_order = ["album", "playlist"]
artist_context_plot = artist_by_context.copy()
artist_context_plot["context"] = pd.Categorical(
    artist_context_plot["context"], categories=context_order, ordered=True
)
artist_context_plot = artist_context_plot.sort_values(["context", "plays"], ascending=[True, True])

q3_context_bars = px.bar(
    artist_context_plot,
    x="plays",
    y="artist",
    color="context",
    orientation="h",
    facet_col="context",
    facet_col_spacing=0.12,
    category_orders={"context": context_order},
    labels={"plays": "Recent Plays", "artist": "Artist", "context": "Listening Context"},
    title="Artist leaders differ by listening mode: album and playlist highlight different favorites",
)

q3_context_bars.update_layout(template="plotly_white", showlegend=False)
q3_context_bars.for_each_annotation(lambda a: a.update(text=a.text.replace("context=", "Context: ")))
q3_context_bars.show()

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.

In [22]:
%pip install plotly kaleido

Note: you may need to restart the kernel to use updated packages.
